# MapBiomas GIF Factory — piloto
Execute em ordem. O resultado em /content é temporário; monte o Drive para preservá-lo.

In [ ]:
REPO_URL = 'https://github.com/mapbiomas/brazil-fire.git'
REPO_REF = 'main'
GEE_PROJECT = 'mapbiomas-brazil'  # Altere para um projeto Earth Engine ao qual você tenha acesso.
if not GEE_PROJECT:
    raise ValueError('Preencha GEE_PROJECT antes de executar.')
import os, subprocess, sys
from google.colab import drive, output
output.enable_custom_widget_manager()
repo_dir = '/content/brazil-fire'
if not os.path.exists(repo_dir):
    subprocess.run(['git', 'clone', REPO_URL, repo_dir], check=True)
    subprocess.run(['git', '-C', repo_dir, 'checkout', '--detach', REPO_REF], check=True)
else:
    print('Diretório já existe: usando seu conteúdo sem substituir arquivos. Confira a versão.')
package_dir = os.path.join(repo_dir, 'utils', 'mapbiomas_gif_factory')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', package_dir], check=True)
source_dir = os.path.join(package_dir, 'src')
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
import ee
from gif_factory.ui import show_ui
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)
# Comente o bloco abaixo para gerar apenas no /content.
try:
    drive.mount('/content/drive')
except Exception as exc:
    print(f'Drive não montado: {exc}. Você ainda pode gerar arquivos em /content.')
print('Preparação concluída.')

In [ ]:
# Camadas personalizadas: copie um objeto e ajuste os valores.
# Obrigatórios: id, name, asset, visualization e legend.
# theme e collection são opcionais e organizam as pastas.
# As bandas são descobertas automaticamente; 'bands' é opcional.
# A camada personalizada é gerada dentro da iniciativa escolhida na interface.
# Exemplo experimental: confirme asset, legenda e resultado no Earth Engine.
LAYERS = [
    {
        'id': 'fogo_anual',
        'name': 'Área queimada anual',
        'theme': 'Fogo',
        'collection': 'Coleção 4',
        'asset': 'projects/mapbiomas-public/assets/brazil/fire/collection4/mapbiomas_fire_collection4_annual_burned_v1',
        # Opcionais: limite o período desta camada; sem eles usa todas as bandas.
        # 'start_year': 2020,
        # 'end_year': 2023,
        'visualization': {
            'min': 0,
            'max': 1,
            'palette': ['fdfdfd', '800000'],
        },
        'legend': [
            {'label': 'Não observado', 'color': 'fdfdfd'},
            {'label': 'Fogo', 'color': '800000'},
        ],
        'status': 'experimental',
    },
    # Exemplo de sequência por profundidade (Solo). Descomente e valide a legenda.
    # {
    #     'id': 'argila_col3',
    #     'name': 'Fração de argila',
    #     'theme': 'Solo',
    #     'collection': 'Coleção 3',
    #     'asset': 'projects/mapbiomas-public/assets/brazil/soil/collection3/mapbiomas_brazil_collection3_soil_clay_fraction_v1',
    #     'axis': 'depth',
    #     'depth_band': 'clay_fraction',
    #     'visualization': {'min': 0, 'max': 100, 'palette': ['ffffcc', '800026']},
    #     'legend': [{'label': '0', 'color': 'ffffcc'}, {'label': '100', 'color': '800026'}],
    # },
    # Adicione outro objeto aqui, separado por vírgula.
]
LAYERS

In [ ]:
# Territórios personalizados: disponíveis com qualquer camada.
# Obrigatórios: id, name e asset de uma FeatureCollection.
# A conta autenticada precisa ter acesso aos assets.
TERRITORIES = [
    {
        'id': 'matopiba',
        'name': 'MATOPIBA',
        'asset': 'projects/mapbiomas-workspace/AUXILIAR/ESTATISTICAS/COLECAO8/VERSAO-1/matopiba',
    },
    {
        'id': 'matopiba_cerrado',
        'name': 'MATOPIBA no Cerrado',
        'asset': 'users/dh-conciani/vectors/matopiba_lapig_cerrado',
    },
    # Exemplo opcional: selecionar um ou mais valores por atributo.
    # {
    #     'id': 'cerrado_caatinga',
    #     'name': 'Cerrado e Caatinga',
    #     'asset': 'projects/ee-ipam/assets/IBGE/limite_biomas_IBGE_2026',
    #     'filter': {'field': 'Bioma', 'values': ['Cerrado', 'Caatinga']},
    # },
]
TERRITORIES

In [ ]:
# Execute novamente após alterar os arrays.
ui = show_ui(LAYERS, TERRITORIES, project=GEE_PROJECT)

In [ ]:
# Depois de selecionar as opções na interface, execute esta célula para gerar o lote.
ui.start_batch()